# Практика к занятию 4: градиентный бустинг

**Что попробуем**:

1. Загрузим датасет с категориальными признаками и подготовим данные.
2. Реализуем простой градиентный бустинг вручную.
3. Сравним с реализациями XGBoost, LightGBM и CatBoost.
4. Проведём оптимизацию гиперпараметров с Optuna.

In [ ]:
# если работаете в colab - тут нет только catboost и optuna
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 6.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from math import log

from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import roc_auc_score, accuracy_score

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import optuna

# Фиксируем seed
RND = 42
np.random.seed(RND)

## 1. Подготовка данных

Для практики будем использовать **датасет Adult (Census Income)** — классический открытый набор данных из UCI Machine Learning Repository.

Он содержит **демографическую информацию о людях из переписи населения США** и используется для задачи **бинарной классификации**:
нужно предсказать, зарабатывает ли человек **более $50 000 в год** на основе его характеристик.

**Описание признаков:**

| Признак          | Тип            | Описание                                                                  |
| ---------------- | -------------- | ------------------------------------------------------------------------- |
| `age`            | числовой       | Возраст человека                                                          |
| `workclass`      | категориальный | Тип занятости (частный сектор, госслужащий, самозанятый и т. д.)          |
| `fnlwgt`         | числовой       | Вес выборки (статистический коэффициент, часто не используется в моделях) |
| `education`      | категориальный | Уровень образования                                                       |
| `education-num`  | числовой       | Количество лет обучения                                                   |
| `marital-status` | категориальный | Семейное положение                                                        |
| `occupation`     | категориальный | Род деятельности / профессия                                              |
| `relationship`   | категориальный | Статус в семье (супруг, ребёнок, не в браке и т. д.)                      |
| `race`           | категориальный | Расовая принадлежность                                                    |
| `sex`            | категориальный | Пол                                                                       |
| `capital-gain`   | числовой       | Доход от капитала                                                         |
| `capital-loss`   | числовой       | Потери капитала                                                           |
| `hours-per-week` | числовой       | Количество рабочих часов в неделю                                         |
| `native-country` | категориальный | Страна происхождения                                                      |
| **`target`**     | бинарный       | Целевая переменная: `1`, если доход > 50 000 $, иначе `0`                 |

Этот датасет хорошо подходит для демонстрации моделей бустинга,
поскольку в нём сочетаются числовые и категориальные признаки, а также умеренный дисбаланс классов.

In [ ]:
df = fetch_openml("adult", version=2, as_frame=True).frame
print(df.shape)
df.head()

(48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
 14  class           48842 non-null  category
dtypes: category(9), int64(6)
memory usage: 2.7 MB


In [ ]:
df.describe()

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week
count,48842.000000,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,38.643585,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,13.710510,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


In [ ]:
df.describe(include=['category'])

,workclass,education,marital-status,occupation,relationship,race,sex,native-country,class
count,46043,48842,48842,46033,48842,48842,48842,47985,48842
unique,8,16,7,14,6,5,2,41,2
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States,<=50K
freq,33906,15784,22379,6172,19716,41762,32650,43832,37155


In [ ]:
print(df['native-country'].unique().to_list())

['United-States', nan, 'Peru', 'Guatemala', 'Mexico', 'Dominican-Republic', 'Ireland', 'Germany', 'Philippines', 'Thailand', 'Haiti', 'El-Salvador', 'Puerto-Rico', 'Vietnam', 'South', 'Columbia', 'Japan', 'India', 'Cambodia', 'Poland', 'Laos', 'England', 'Cuba', 'Taiwan', 'Italy', 'Canada', 'Portugal', 'China', 'Nicaragua', 'Honduras', 'Iran', 'Scotland', 'Jamaica', 'Ecuador', 'Yugoslavia', 'Hungary', 'Hong', 'Greece', 'Trinadad&Tobago', 'Outlying-US(Guam-USVI-etc)', 'France', 'Holand-Netherlands']


/tmp/ipython-input-4195455339.py:1: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  print(df['native-country'].unique().to_list())


In [ ]:
df['native-country'].value_counts().sort_values()

,count
native-country,
Holand-Netherlands,1
Hungary,19
Honduras,20
Scotland,21
Laos,23
Yugoslavia,23
Outlying-US(Guam-USVI-etc),23
Trinadad&Tobago,27
Cambodia,28


In [ ]:
# Простейшая очистка: заменим '?' на NaN и удалим пропуски
df = df.replace('?', np.nan).dropna()

In [ ]:
# Целевая переменная: доход >50K — 1, иначе 0
df['target'] = df['class'].apply(lambda x: 1 if '>50K' in x else 0)
df = df.drop(columns=['class'])

In [ ]:
# Определим категориальные колонки
cat_cols = df.select_dtypes(include=['category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'target']
print("Категориальные признаки:", cat_cols)

Категориальные признаки: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [ ]:
# Разделим данные на train/val/test (60/20/20)
df_trainval, df_test = train_test_split(
    df, test_size=0.2, random_state=RND, stratify=df['target']
)
df_train, df_val = train_test_split(
    df_trainval, test_size=0.25, random_state=RND, stratify=df_trainval['target']
)

X_train, y_train = df_train.drop(columns=['target']), df_train['target'].values
X_val,   y_val   = df_val.drop(columns=['target']), df_val['target'].values
X_test,  y_test  = df_test.drop(columns=['target']), df_test['target'].values

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (29304, 14), Val: (9769, 14), Test: (9769, 14)


In [ ]:
y_train, y_val, y_test = y_train.astype(int), y_val.astype(int), y_test.astype(int)

In [ ]:
# One-hot кодирование
X_train = pd.get_dummies(X_train, drop_first=True)
X_val   = pd.get_dummies(X_val,   drop_first=True)
X_test  = pd.get_dummies(X_test,  drop_first=True)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (29304, 97), Val: (9769, 97), Test: (9769, 97)


## 2. Простейшая реализация градиентного бустинга

Идея (в упрощённой форме):
1. Храним **сырые предсказания** $F(x)$ вместо вероятностей. Начальное значение — `logit(mean(y))`.
2. На каждой итерации вычисляем *псевдо-остатки* (negative gradient) для логистической ошибки: `r = y - p`,
   где `p = sigmoid(F)`.
3. Обучаем `DecisionTreeRegressor` на `r` и добавляем предсказание дерева к `F` с множителем `learning_rate`.

Это **не** полностью оптимизированная реализация (в продакшене используют более сложные трюки),
но чисто обучающий пример, который показывает логику алгоритма.

In [ ]:
# help(np.clip)

In [ ]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    """Сигмоида (элементно)."""
    return 1.0 / (1.0 + np.exp(-x))

def _init_F0(y: np.ndarray, eps: float = 1e-6) -> float:
    """Вычислить начальный F0 = logit(mean(y)) с клиппингом."""
    p0 = np.clip(np.mean(y), eps, 1 - eps)
    return np.log(p0 / (1 - p0))

In [ ]:
def fit_gradient_boosting(
    X,
    y,
    n_estimators: int = 100,
    learning_rate: float = 0.1,
    max_depth: int = 3,
    min_samples_leaf: int = 5,
    verbose: bool = False
):
    """
    Обучить "ручной" градиентный бустинг для бинарной классификации.
    Возвращает словарь-модель: {'F0': float, 'trees': list[DecisionTreeRegressor], 'learning_rate': float}
    """
    X_arr = np.asarray(X)
    y_arr = np.asarray(y)

    F0 = _init_F0(y_arr)
    F = np.full(len(y_arr), fill_value=F0, dtype=float)

    trees = []
    for m in range(n_estimators):
        p = sigmoid(F)               # текущие вероятности
        residual = y_arr - p         # negative gradient для log-loss

        tree = DecisionTreeRegressor(
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=0
        )
        tree.fit(X_arr, residual)    # учим дерево предсказывать остатки
        update = tree.predict(X_arr)
        F = F + learning_rate * update

        trees.append(tree)

        if verbose and (m + 1) % 10 == 0:
            auc = roc_auc_score(y_arr, sigmoid(F))
            print(f"iter={m+1}  AUC={auc:.4f}")

    model = {
        "F0": float(F0),
        "trees": trees,
        "learning_rate": float(learning_rate)
    }
    return model

In [ ]:
def decision_function(model, X):
    """
    Возвращает сырые предсказания F (логиты).
    model: словарь из fit_gradient_boosting
    """
    X_arr = np.asarray(X)
    F = np.full(X_arr.shape[0], fill_value=model["F0"], dtype=float)
    for tree in model["trees"]:
        F += model["learning_rate"] * tree.predict(X_arr)
    return F

def predict_proba(model, X):
    """
    Возвращает массив shape=(n_samples, 2): [prob(0), prob(1)].
    """
    F = decision_function(model, X)
    p = sigmoid(F)
    return np.vstack([1 - p, p]).T

def predict(model, X, threshold: float = 0.5):
    """Классы 0/1 по порогу."""
    probs = predict_proba(model, X)[:, 1]
    return (probs >= threshold).astype(int)

In [ ]:
# Обучаем
model = fit_gradient_boosting(
    X_train, y_train,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    min_samples_leaf=5,
    verbose=True
)

# Предсказываем
probs = predict_proba(model, X_test)[:, 1]
preds = predict(model, X_test)

print('Manual GB AUC:', roc_auc_score(y_test, probs))
print('Manual GB Accuracy:', accuracy_score(y_test, preds))

iter=10  AUC=0.8654
iter=20  AUC=0.8765
iter=30  AUC=0.8841
iter=40  AUC=0.8890
iter=50  AUC=0.8913
iter=60  AUC=0.8951
iter=70  AUC=0.8987
iter=80  AUC=0.9002
iter=90  AUC=0.9025
iter=100  AUC=0.9040
Manual GB AUC: 0.9032926706711153
Manual GB Accuracy: 0.8564847988535162


## 3. Быстрый сравнительный анализ: XGBoost / LightGBM / CatBoost

Посмотрим на три популярные реализации градиентного бустинга. Если пакеты не установлены, запустите ячейку с `pip install` из начала ноутбука.
Далее сравним метрики (AUC, accuracy).

In [ ]:
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_test  = lgb.Dataset(X_test, label=y_test, reference=lgb_train)

params = {'objective': 'binary', 'metric': 'auc', 'seed': RND}
gbm = lgb.train(params, lgb_train, num_boost_round=100)

preds_train = gbm.predict(X_train)
preds_test = gbm.predict(X_test)
print("LightGBM AUC train:", roc_auc_score(y_train, preds_train))
print("LightGBM AUC test:", roc_auc_score(y_test, preds_test))
print("LightGBM Accuracy:", accuracy_score(y_test, (preds>=0.5).astype(int)))

[LightGBM] [Info] Number of positive: 7012, number of negative: 22292
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007462 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 755
[LightGBM] [Info] Number of data points in the train set: 29304, number of used features: 85
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.239285 -> initscore=-1.156605
[LightGBM] [Info] Start training from score -1.156605
LightGBM AUC train: 0.9457151503065314
LightGBM AUC test: 0.929082661713887
LightGBM Accuracy: 0.8564847988535162


In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test, label=y_test)

params = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'seed': RND}
bst = xgb.train(params, dtrain, num_boost_round=50)

preds_train = bst.predict(dtrain)
preds_test = bst.predict(dtest)
print("XGBoost AUC train:", roc_auc_score(y_train, preds_train))
print("XGBoost AUC test:", roc_auc_score(y_test, preds_test))
print("XGBoost Accuracy:", accuracy_score(y_test, (preds>=0.5).astype(int)))

XGBoost AUC train: 0.9477202010672227
XGBoost AUC test: 0.9288032159914553
XGBoost Accuracy: 0.8756269833145665


In [ ]:
cat_cols

['workclass',
 'education',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'sex',
 'native-country']

In [ ]:
train_pool = cb.Pool(X_train, y_train)#, cat_features=cat_cols)
test_pool  = cb.Pool(X_test,  y_test)#,  cat_features=cat_cols)

cb = cb.CatBoostClassifier(iterations=200, learning_rate=0.1, depth=6, verbose=False, random_seed=RND)
cb.fit(train_pool)

preds = cb.predict_proba(test_pool)[:,1]

preds_train = cb.predict_proba(train_pool)[:,1]
preds_test = cb.predict_proba(test_pool)[:,1]
print("CatBoost AUC train:", roc_auc_score(y_train, preds_train))
print("CatBoost AUC test:", roc_auc_score(y_test, preds_test))
print("CatBoost Accuracy:", accuracy_score(y_test, (preds_test>=0.5).astype(int)))

CatBoost AUC train: 0.940152127254818
CatBoost AUC test: 0.9288652063195831
CatBoost Accuracy: 0.8762411710512846


In [ ]:
cb.get_feature_importance(prettified=True).head()

,Feature Id,Importances
0,marital-status_Married-civ-spouse,24.733664
1,capital-gain,21.055755
2,age,11.035784
3,education-num,10.019589
4,capital-loss,8.866403


In [ ]:
# help(cb.get_feature_importance)

In [ ]:
cb.get_feature_importance(prettified=True)['Importances'].sum()

np.float64(99.99999999999997)

## 4. Бонус*: оптимизация гиперпараметров CatBoost с Optuna

Ниже — компактный пример того, как можно оптимизировать несколько важных гиперпараметров CatBoost
с помощью Optuna. В рабочей среде можно увеличить число итераций (итераций Optuna) и использовать
k-fold CV вместо простой валидации.


In [ ]:
def objective(trial):
    params = {
        'iterations': 200,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'random_seed': RND,
        'verbose': False
    }
    model = cb.CatBoostClassifier(**params)
    # Разделим тренировочный набор на подвалидацию для Optuna
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=RND)
    pool_tr = cb.Pool(X_tr, y_tr) #, cat_features=cat_cols)
    pool_val = cb.Pool(X_val, y_val) #, cat_features=cat_cols)
    model.fit(pool_tr)
    preds = model.predict_proba(pool_val)[:,1]
    return roc_auc_score(y_val, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)
print('Best trial:', study.best_trial.params)

[I 2025-11-07 21:45:10,166] A new study created in memory with name: no-name-b489fc06-deaf-4766-bdbf-0d61fcf88fdb
[I 2025-11-07 21:45:13,724] Trial 0 finished with value: 0.9076839083832278 and parameters: {'learning_rate': 0.010828570529191776, 'depth': 6, 'l2_leaf_reg': 4.0990824728985995}. Best is trial 0 with value: 0.9076839083832278.
[I 2025-11-07 21:45:15,564] Trial 1 finished with value: 0.9168566919189647 and parameters: {'learning_rate': 0.03473195284928244, 'depth': 6, 'l2_leaf_reg': 5.987772829038667}. Best is trial 1 with value: 0.9168566919189647.
[I 2025-11-07 21:45:17,741] Trial 2 finished with value: 0.911937245945478 and parameters: {'learning_rate': 0.014538560637654286, 'depth': 7, 'l2_leaf_reg': 1.8093012013467882}. Best is trial 1 with value: 0.9168566919189647.
[I 2025-11-07 21:45:20,425] Trial 3 finished with value: 0.9224356761220882 and parameters: {'learning_rate': 0.11379053254865965, 'depth': 8, 'l2_leaf_reg': 2.12519472856809}. Best is trial 3 with value: 

Best trial: {'learning_rate': 0.22339082865366805, 'depth': 4, 'l2_leaf_reg': 0.01340653855214032}


PS: лучше для валидации использовать отдельный набор данных (т.е. суммарно разбивать исходный датасет на 3 выборки: тренировочную для обучения, валидационную для подбора гиперпараметров и тестовую для чистой и честной оценки качества).